In [0]:
# Read from the Bronze Delta Lake table
df_directors= (
    spark.read
    .format("delta")
    .load("abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_directors")
)


In [0]:
display(df_directors)

In [0]:
from pyspark.sql import functions as F

dir_col = F.col("director")
dir_trimmed = F.trim(dir_col)

# 1. Single-pass summary of text quality issues in 'director'
df_directors.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(dir_trimmed).alias("distinct_directors"),
    F.sum(F.when(dir_col.isNull() | (dir_trimmed == ""), 1).otherwise(0)).alias("dq_null_or_blank"),
    F.sum(F.when(dir_col != dir_trimmed, 1).otherwise(0)).alias("dq_outer_spaces"),
    F.sum(F.when(dir_col.rlike(r"\s{2,}"), 1).otherwise(0)).alias("dq_double_spaces"),
    F.sum(F.when(dir_col.contains(","), 1).otherwise(0)).alias("dq_contains_comma"),
    F.sum(F.when(dir_trimmed.rlike(r"^[0-9\W_]+$"), 1).otherwise(0)).alias("dq_numeric_or_symbol_only"),
    F.sum(F.when(F.lower(dir_trimmed).isin("unknown", "n/a", "na", "none", "null", "tbd"), 1).otherwise(0)).alias("dq_placeholder_text")
).display()

# 2. Inspect any suspicious rows (commas, numeric/symbols, placeholders, or shortest names)
(
    df_directors
    .filter(
        dir_col.isNull() | (dir_trimmed == "") |
        dir_col.contains(",") |
        dir_trimmed.rlike(r"^[0-9\W_]+$") |
        F.lower(dir_trimmed).isin("unknown", "n/a", "na", "none", "null", "tbd") |
        (F.length(dir_trimmed) <= 3)
    )
    .select("show_id", "director", F.length(dir_trimmed).alias("char_len"))
    .orderBy("char_len", "director")
    .display()
)


In [0]:
from pyspark.sql import functions as F

# 1. Clean whitespace (trim outer spaces, collapse repeated internal spaces, blank -> NULL)
staged_directors_df = df_directors.withColumn(
    "director",
    F.when(
        F.col("director").isNull() | (F.trim(F.col("director")) == ""),
        F.lit(None).cast("string")
    ).otherwise(
        F.trim(F.regexp_replace(F.col("director"), r"\s+", " "))
    )
)

# 2. Quarantine invalid director values (NULL, blank, or numeric/symbol-only junk)
is_invalid_director = (
    F.col("director").isNull() |
    F.col("director").rlike(r"^[0-9\W_]+$")
)

quarantine_director_col_df = staged_directors_df.filter(is_invalid_director)
clean_directors_df = staged_directors_df.filter(~is_invalid_director)

# 3. Post-transformation validation check
clean_directors_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("director").alias("distinct_directors"),
    F.sum(F.when(F.col("director").isNull() | (F.trim(F.col("director")) == ""), 1).otherwise(0)).alias("val_null_or_blank"),
    F.sum(F.when(F.col("director") != F.trim(F.col("director")), 1).otherwise(0)).alias("val_outer_spaces"),
    F.sum(F.when(F.col("director").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces")
).display()


xx

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load valid show_ids from your clean Silver netflix_titles table
silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

id_str = F.trim(F.col("show_id").cast("string"))

# 1. Audit show_id quality AND composite duplicate pairs in one pass
clean_directors_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(id_str.isNull() | (id_str == ""), 1).otherwise(0)).alias("dq_null_show_id"),
    F.sum(F.when(id_str.isNotNull() & ~id_str.rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("dq_non_digit_show_id"),
    F.countDistinct(id_str).alias("distinct_show_ids"),
    (F.count("*") - F.countDistinct(id_str, F.lower(F.col("director")))).alias("dq_duplicate_pairs")
).display()

# 2. Check for orphan show_ids (IDs in netflix_directors missing from Silver netflix_titles)
orphan_directors_df = (
    clean_directors_df
    .withColumn("show_id_clean", id_str)
    .join(silver_titles_ids, F.col("show_id_clean") == silver_titles_ids.show_id, "left_anti")
)

print(f"Orphan show_id rows in netflix_directors: {orphan_directors_df.count()}")
orphan_directors_df.display()


In [0]:
from pyspark.sql import functions as F

df_directors = spark.read.format("delta").load(
    "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_directors"
)

silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

if "_rescued_data" not in df_directors.columns:
    df_directors = df_directors.withColumn("_rescued_data", F.lit(None).cast("string"))

# 1. Standardise show_id (trimmed STRING) and director (collapsed whitespace, blank -> NULL)
staged_directors = (
    df_directors
    .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
    .withColumn(
        "director",
        F.when(F.col("director").isNull() | (F.trim(F.col("director")) == ""), F.lit(None).cast("string"))
         .otherwise(F.trim(F.regexp_replace(F.col("director"), r"\s+", " ")))
    )
)

# 2. Check foreign-key match against Silver netflix_titles
staged_with_fk = (
    staged_directors.alias("d")
    .join(silver_titles_ids.alias("t"), F.col("d.show_id") == F.col("t.show_id"), "left")
    .select(
        F.col("d.*"),
        F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
    )
)

# 3. Unified quarantine condition (corrupt CSV, invalid show_id, invalid director, or orphan foreign key)
invalid_condition = (
    F.col("_rescued_data").isNotNull() |
    F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
    F.col("director").isNull() | F.col("director").rlike(r"^[0-9\W_]+$") |
    F.col("_is_orphan")
)

# 4. Build Quarantine DataFrame with explicit failure reasons
quarantine_directors_df = (
    staged_with_fk
    .filter(invalid_condition)
    .withColumn(
        "quarantine_reason",
        F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
         .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
         .when(F.col("director").isNull() | F.col("director").rlike(r"^[0-9\W_]+$"), "Invalid or Missing director")
         .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
         .otherwise("Unknown validation failure")
    )
    .drop("_is_orphan")
    .withColumn("_quarantined_at", F.current_timestamp())
)

# 5. Build Clean Silver DataFrame (deduplicated on composite key show_id + director)
clean_directors_df = (
    staged_with_fk
    .filter(~invalid_condition)
    .drop("_is_orphan", "_rescued_data")
    .dropDuplicates(["show_id", "director"])
    .withColumn("_silver_processed_at", F.current_timestamp())
)


In [0]:
bronze_cnt     = df_directors.count()
quarantine_cnt = quarantine_directors_df.count()
clean_cnt      = clean_directors_df.count()
dedup_cnt      = bronze_cnt - quarantine_cnt - clean_cnt

print("=" * 55)
print("📊 SILVER RECONCILIATION (netflix_directors)")
print("=" * 55)
print(f"Bronze rows read:         {bronze_cnt:,}")
print(f"Quarantined rows:         {quarantine_cnt:,}")
print(f"Duplicate pairs dropped:  {dedup_cnt:,}")
print(f"Clean Silver rows:        {clean_cnt:,}")
print("=" * 55)

# Hard quality assertions
assert clean_cnt > 0, "Quality Gate Failed: clean_directors_df is empty!"
assert clean_cnt == clean_directors_df.select("show_id", "director").distinct().count(), "Duplicate (show_id, director) pairs remain!"


In [0]:
display(quarantine_directors_df)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_directors"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_directors"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_directors"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_directors"

def process_directors_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    # Read valid parent show_ids from Silver netflix_titles for foreign-key check
    silver_titles_ids = spark.read.format("delta").load(
        "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
    ).select("show_id").distinct()

    # 1. Standardise strings
    staged = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "director",
            F.when(F.col("director").isNull() | (F.trim(F.col("director")) == ""), F.lit(None).cast("string"))
             .otherwise(F.trim(F.regexp_replace(F.col("director"), r"\s+", " ")))
        )
    )

    # 2. Check foreign-key match against Silver netflix_titles
    staged_with_fk = (
        staged.alias("d")
        .join(silver_titles_ids.alias("t"), F.col("d.show_id") == F.col("t.show_id"), "left")
        .select(
            F.col("d.*"),
            F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
        )
    )

    # 3. Validation rules
    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("director").isNull() | F.col("director").rlike(r"^[0-9\W_]+$") |
        F.col("_is_orphan")
    )

    quarantine_df = (
        staged_with_fk.filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("director").isNull() | F.col("director").rlike(r"^[0-9\W_]+$"), "Invalid or Missing director")
             .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
             .otherwise("Unknown validation failure")
        )
        .drop("_is_orphan")
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_with_fk.filter(~invalid_condition)
        .drop("_is_orphan", "_rescued_data")
        .dropDuplicates(["show_id", "director"])
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 4. Write only new batch to Silver (Upsert) and Quarantine (Append)
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(
                    clean_df.alias("source"),
                    "target.show_id = source.show_id AND target.director = source.director"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

# Stream ONLY new Bronze commits using Structured Streaming
query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_directors_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("✅ Incremental Silver processing complete for netflix_directors.")
